In [2]:
import pandas as pd
import altair as alt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
df = pd.read_csv('data.csv')

In [4]:
# Make a dataframe for each decade
twenties_df = df[(df['year'] >= 1920) & (df['year'] <= 1929)]
thirties_df = df[(df['year'] >= 1930) & (df['year'] <= 1939)]
forties_df = df[(df['year'] >= 1940) & (df['year'] <= 1949)]
fifties_df = df[(df['year'] >= 1950) & (df['year'] <= 1959)]
sixties_df = df[(df['year'] >= 1960) & (df['year'] <= 1969)]
seventies_df = df[(df['year'] >= 1970) & (df['year'] <= 1979)]
eighties_df = df[(df['year'] >= 1980) & (df['year'] <= 1989)]
nineties_df = df[(df['year'] >= 1990) & (df['year'] <= 1999)]
aughts_df = df[(df['year'] >= 2000) & (df['year'] <= 2009)]
tens_df = df[(df['year'] >= 2010) & (df['year'] <= 2019)]
now_df = df[df['year'] >= 2020]

In [5]:
# Make each decade dataframe only the top 10% popularity scores
twenties_df = twenties_df[twenties_df['popularity'] >= twenties_df['popularity'].quantile(0.9)]
thirties_df = thirties_df[thirties_df['popularity'] >= thirties_df['popularity'].quantile(0.9)]
forties_df = forties_df[forties_df['popularity'] >= forties_df['popularity'].quantile(0.9)]
fifties_df = fifties_df[fifties_df['popularity'] >= fifties_df['popularity'].quantile(0.9)]
sixties_df = sixties_df[sixties_df['popularity'] >= sixties_df['popularity'].quantile(0.9)]
seventies_df = seventies_df[seventies_df['popularity'] >= seventies_df['popularity'].quantile(0.9)]
eighties_df = eighties_df[eighties_df['popularity'] >= eighties_df['popularity'].quantile(0.9)]
nineties_df = nineties_df[nineties_df['popularity'] >= nineties_df['popularity'].quantile(0.9)]
aughts_df = aughts_df[aughts_df['popularity'] >= aughts_df['popularity'].quantile(0.9)]
tens_df = tens_df[tens_df['popularity'] >= tens_df['popularity'].quantile(0.9)]
now_df = now_df[now_df['popularity'] >= now_df['popularity'].quantile(0.9)]

In [6]:
# Make dataframe of the mean qualities for each decade

decade_dfs = [twenties_df, thirties_df, forties_df, fifties_df, sixties_df, seventies_df, eighties_df, nineties_df, aughts_df, tens_df, now_df]
decades = ['1920s', '1930s', '1940s', '1950s', '1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s']
qualities = ['speechiness', 'acousticness', 'liveness', 'danceability', 'valence', 'instrumentalness']

decadesmeans = {}
# Iterate through decades using i to reference decade_dfs and decades lists
for i in range(len(decades)):
    # Create a dictionary of the means for the decade
    row = {}
    for quality in qualities:
            row[quality] = decade_dfs[i][quality].mean()
    # Append decade means dict to a dict of dicts of each decade, using decade as key
    decadesmeans[decades[i]] = row

# Convert dict of dicts to dataframe and convert quality from index to a column
decadesmeans_df = pd.DataFrame(decadesmeans).reset_index(names='quality')
# Melt dataframe for plotting
decadesmeans_df = decadesmeans_df.melt(id_vars=['quality'], var_name='decade')
decadesmeans_df

,quality,decade,value
0,speechiness,1920s,0.087582
1,acousticness,1920s,0.942866
2,liveness,1920s,0.199008
3,danceability,1920s,0.612006
4,valence,1920s,0.607642
...,...,...,...
61,acousticness,2020s,0.208012
62,liveness,2020s,0.169086
63,danceability,2020s,0.720524
64,valence,2020s,0.522978


In [18]:
# Create radio selection for decades
radio = alt.binding_radio(options = decades, name = 'Decades: ')
selection = alt.selection_point(fields = ['decade'], bind = radio, value = '2020s')

# Define a custom color scale to make each bar a different color to match the other figures
color_scale = alt.Scale(range=['#4C78A8', '#F58517', '#72B7B2', '#53A24A', '#EECA3B', '#B379A2'])

# Create the bar plot as a overlapping bar plot of each decades qualities, only showing selected decade
chart = alt.Chart(decadesmeans_df).mark_bar().encode(
    alt.Y(
        'quality:O',
        title='Musical Qualities',
        sort=['speechiness', 'acousticness', 'liveness', 'danceability', 'valence', 'instrumentalness']),
    alt.X(
        'value:Q',
        title='Average Value',
        stack=None,
        scale=alt.Scale(domain=[0, 1])),
    color = alt.condition(selection, alt.Color('quality:O', scale=color_scale, legend=None), alt.value(None))
).properties(
    title='Average Musical Qualities of Top 10% of Songs of the Decade',
    height=300, width=500
).add_params(selection)

chart

alt.Chart(...)

In [8]:
# Save figure as html file
chart.save('decades_top_songs_qualities.html')